# The JAX backend

The unicycle specification of Section V-A of the paper in the JAX evaluator, against the Warp evaluator and the Warp chain. JAX and Warp on the CPU in float64; needs the extra `jax`; about 3.5 minutes on an AMD Ryzen 9 7950X.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import warp as wp
from IPython.display import Markdown

from sparsemax_dstl.jax import lower_max, lower_min, robustness
from sparsemax_dstl.jax.methods import SEMANTICS
from sparsemax_dstl.stl import compile_formula
from sparsemax_dstl.tasks import planar_al, planar_disk_jax, planar_jax
from sparsemax_dstl.tasks import planar_disk as D
from sparsemax_dstl.tasks.planar_al_jax import JaxChain
from sparsemax_dstl.tasks.planar_warp import WarpChain
from sparsemax_dstl.warp import Evaluator, matched_param

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platforms", "cpu")
wp.config.log_level = wp.LOG_WARNING


def table(header, rows):
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(x) for x in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def shown(x):
    if abs(x) >= 0.001:
        return round(x, 4)
    return "< 0.001" if x >= 0 else "> -0.001"

The specification of Equation (18) compiled once for its value at $t = 0$, and the predicate values of $S_1$ and $S_2$ at the 3 s wait.

In [2]:
eps = 0.1
u1, u2, tm = D.trajectories(30, eps)
T = tm["T"]
spec, conj, until = D.specification(tm["a1"], tm["b1"], tm["a2"], tm["b2"], T)
prog = compile_formula(spec, T, reads=[(spec, [0])])
z = jax.vmap(lambda u: planar_jax.rollout(D.Z0, u))(jnp.stack([u1, u2]))
S = planar_disk_jax.scores(z[..., :2], D.REGIONS)
print("predicate values", S.shape, S.dtype, "on", jax.devices()[0])

predicate values (2, 447, 8) float64 on cpu:0


The robustness under every measure, with every smoothing at the error $\varepsilon = 0.1$ m per node (Equation (15)): Table I, columns "value".

In [3]:
measures = {"exact": "exact", "lse_plain": "plain LSE", "lse": "sound LSE", "gm_pm01": "GMR (0, 1)", "gm_pm10": "GMR (-10, 10)", "sparsemax": "sparsemax"}
smooth = [m for m in measures if m != "exact"]


def rho(S, measure):
    return robustness(prog, S, SEMANTICS[measure], eps)[..., 0]


table(["measure", "$S_1$", "$S_2$"], [[measures[m]] + [round(float(x), 4) for x in rho(S, m)] for m in measures])

| measure | $S_1$ | $S_2$ |
|---|---|---|
| exact | -0.015 | 0.3 |
| plain LSE | 0.024 | 0.3572 |
| sound LSE | -0.076 | 0.2575 |
| GMR (0, 1) | -0.0001 | 1.6855 |
| GMR (-10, 10) | -0.0092 | 0.4927 |
| sparsemax | -0.1096 | 0.2137 |

`lower_min` and `lower_max` (Equation (7)) with $\gamma = 2\varepsilon / (1 - 1/m)$ on the $m$ values of $\neg\mathrm{Red}$ that the until reads on $S_1$; their derivative is the sparsemax weight.

In [4]:
x = -S[0, : tm["b1"] + 1, 0]
gamma = 2 * eps / (1 - 1 / len(x))
p = jax.grad(lower_min)(x, gamma)
q = jax.grad(lower_max)(x, gamma)
print("m =", len(x), " gamma =", round(gamma, 4))
print("min", round(float(x.min()), 4), " lower_min", round(float(lower_min(x, gamma)), 4))
print("weights of lower_min:", int((p != 0).sum()), "nonzero; their sum on the samples inside Red", round(float(p[x < 0].sum()), 4))
print("max", round(float(x.max()), 4), " lower_max", round(float(lower_max(x, gamma)), 4))
print("weights of lower_max:", int((q != 0).sum()), "nonzero; their sum", round(float(q.sum()), 4))

m = 257  gamma = 0.2008
min -0.015  lower_min -0.1096
weights of lower_min: 21 nonzero; their sum on the samples inside Red 1.0


max 2.4079  lower_max 2.3961
weights of lower_max: 41 nonzero; their sum 1.0


Value and gradient with respect to the predicate values on $S_1$ and $S_2$: the largest difference between the JAX and the Warp evaluator, in units of $10^{-15}$, and the number of nonzero entries of the gradient.

In [5]:
Sw = wp.array(np.asarray(S), dtype=wp.float64, device="cpu")


def compare(measure):
    r, G = jax.vmap(jax.value_and_grad(lambda s: rho(s, measure)))(S)
    param = None if measure == "exact" else matched_param(prog, measure, eps)
    rw, Gw = Evaluator(prog, measure, param, 2, wp.float64, "cpu").gradient(Sw)
    dr, dG = np.abs(r - rw.numpy()[:, 0]).max(), np.abs(G - Gw.numpy()).max()
    return [round(float(dr) * 1e15, 2), round(float(dG) * 1e15, 2), int((G != 0).sum()), int((Gw.numpy() != 0).sum())]


table(["measure", "value", "gradient", "nonzero, JAX", "nonzero, Warp"], [[measures[m]] + compare(m) for m in measures])

| measure | value | gradient | nonzero, JAX | nonzero, Warp |
|---|---|---|---|---|
| exact | 0.0 | 0.0 | 2 | 2 |
| plain LSE | 0.01 | 0.04 | 5188 | 5188 |
| sound LSE | 0.0 | 0.04 | 5188 | 5188 |
| GMR (0, 1) | 1.55 | 0.02 | 2615 | 2615 |
| GMR (-10, 10) | 0.0 | 0.13 | 2615 | 2615 |
| sparsemax | 0.06 | 0.03 | 34 | 34 |

The optimization of Table I on the JAX chain and on the Warp chain: 400 updates of the augmented Lagrangian solver from $S_1$ and $S_2$, for $c = 0$ and $c = 0.2$ m.

In [6]:
V0 = np.stack([u1, u2]) / D.U_MAX
levels = (0.0, 0.2)


def optimize(Chain, measure, c):
    chain = Chain(conj, T, measure, eps, D.Z0, D.REGIONS, 2)
    return planar_al.solve(chain, V0, budget=400, lam=0.01, delta=c, alpha0=0.002)[0]


Vj = {(m, c): optimize(JaxChain, m, c) for m in smooth for c in levels}
Vw = {(m, c): optimize(WarpChain, m, c) for m in smooth for c in levels}

Exact robustness $\rho$ of the returned trajectories on each chain (a value in $(0, 0.001)$ is shown as < 0.001, as in Table I), and the largest difference between the two chains' commands $u_t / u_{\max}$ and positions. Table I's "returned $\rho$" holds the rows from $S_1$ on the JAX chain.

In [7]:
def row(m, c, g):
    zj, sj = D.summary(Vj[m, c][g], tm, eps)
    zw, sw = D.summary(Vw[m, c][g], tm, eps)
    dV, dz = np.abs(Vj[m, c][g] - Vw[m, c][g]).max(), np.abs(zj[:, :2] - zw[:, :2]).max()
    return [measures[m], c, ("$S_1$", "$S_2$")[g], shown(sj["exact"]), shown(sw["exact"]), round(float(dV), 6), round(float(dz), 6)]


table(["measure", "$c$", "from", "$\\rho$, JAX chain", "$\\rho$, Warp chain", "commands", "positions (m)"], [row(m, c, g) for m in smooth for c in levels for g in (0, 1)])

| measure | $c$ | from | $\rho$, JAX chain | $\rho$, Warp chain | commands | positions (m) |
|---|---|---|---|---|---|---|
| plain LSE | 0.0 | $S_1$ | -0.0584 | -0.0586 | 0.000498 | 0.007925 |
| plain LSE | 0.0 | $S_2$ | -0.0933 | -0.0933 | 6e-06 | 0.000149 |
| plain LSE | 0.2 | $S_1$ | 0.1458 | 0.1461 | 0.000289 | 0.0044 |
| plain LSE | 0.2 | $S_2$ | 0.1103 | 0.1102 | 8e-06 | 8.9e-05 |
| sound LSE | 0.0 | $S_1$ | 0.0451 | 0.0445 | 0.001034 | 0.015239 |
| sound LSE | 0.0 | $S_2$ | 0.0377 | 0.0343 | 0.00024 | 0.004937 |
| sound LSE | 0.2 | $S_1$ | 0.218 | 0.2183 | 3e-05 | 0.000424 |
| sound LSE | 0.2 | $S_2$ | 0.2482 | 0.2471 | 0.000129 | 0.001793 |
| GMR (0, 1) | 0.0 | $S_1$ | < 0.001 | < 0.001 | 0.0 | 0.0 |
| GMR (0, 1) | 0.0 | $S_2$ | < 0.001 | < 0.001 | 0.0 | 0.0 |
| GMR (0, 1) | 0.2 | $S_1$ | < 0.001 | < 0.001 | 0.0 | 0.0 |
| GMR (0, 1) | 0.2 | $S_2$ | < 0.001 | < 0.001 | 0.0 | 0.0 |
| GMR (-10, 10) | 0.0 | $S_1$ | < 0.001 | < 0.001 | 0.000478 | 0.007651 |
| GMR (-10, 10) | 0.0 | $S_2$ | > -0.001 | > -0.001 | 1e-06 | 1.5e-05 |
| GMR (-10, 10) | 0.2 | $S_1$ | 0.1413 | 0.1415 | 0.000132 | 0.002682 |
| GMR (-10, 10) | 0.2 | $S_2$ | 0.1463 | 0.1464 | 2.1e-05 | 0.000798 |
| sparsemax | 0.0 | $S_1$ | 0.0866 | 0.0866 | 0.0 | 0.0 |
| sparsemax | 0.0 | $S_2$ | 0.0878 | 0.0876 | 0.000186 | 0.003731 |
| sparsemax | 0.2 | $S_1$ | 0.248 | 0.2482 | 2.1e-05 | 0.000446 |
| sparsemax | 0.2 | $S_2$ | 0.2892 | 0.2894 | 0.000162 | 0.002146 |